# Notebook 2 (simple, testable): pause for human approval, resume, and recall a returning user

**Outcome:** a paused-and-resumed run that clears a human approval step and recalls a returning user from memory.

Plain Python, no framework. Everything is explicit so you can see how it works:

| Concept | Implementation here |
|---|---|
| Pause | When the model calls `request_approval`, the loop saves the full conversation to `sessions/<id>.json` and **returns** |
| Resume | `resume()` reloads that file, injects the human's decision as the tool result, and continues the loop |
| Approval guard | `book_trip` refuses unless the session was approved (the model cannot talk its way past it) |
| Memory | `memory.json` keyed by user, with `save_memory` / `recall_memory` tools, shared across sessions |

## 0. Setup (pick one)

**Option A: no model at all (`BACKEND = "mock"`).** A scripted fake LLM drives the agent loop. This tests *your harness* (retries, pause/resume, memory) deterministically. Start here.

**Option B: free open-source model via Ollama (`BACKEND = "ollama"`).**
1. Install Ollama: https://ollama.com
2. `ollama pull qwen2.5:7b` (or `llama3.1:8b`; the model must support tool calling)
3. Keep Ollama running (it serves an OpenAI-compatible API at `http://localhost:11434/v1`)

Any other OpenAI-compatible free endpoint (Groq free tier, OpenRouter free models, LM Studio, vLLM) works by changing `base_url`, `api_key` and `MODEL`.

In [ ]:
%pip install -q openai
import json, os, shutil
from openai import OpenAI

BACKEND = "mock"        # "mock" or "ollama"
MODEL = "qwen2.5:7b"
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

SESS_DIR, MEM_FILE = "sessions", "memory.json"
def reset_storage():
    shutil.rmtree(SESS_DIR, ignore_errors=True); os.makedirs(SESS_DIR)
    if os.path.exists(MEM_FILE): os.remove(MEM_FILE)
reset_storage()

## 1. Storage and tools

In [ ]:
def load_mem():
    return json.load(open(MEM_FILE)) if os.path.exists(MEM_FILE) else {}

def save_session(s):  json.dump(s, open(f"{SESS_DIR}/{s['id']}.json", "w"), indent=1)
def load_session(sid): return json.load(open(f"{SESS_DIR}/{sid}.json"))

# --- tools: each takes ctx (runtime info the model can't forge) plus the model's arguments
def save_memory(ctx, key, value):
    mem = load_mem(); mem.setdefault(ctx["user_id"], {})[key] = value
    json.dump(mem, open(MEM_FILE, "w"), indent=1)
    return {"status": "saved", "key": key}

def recall_memory(ctx, query=""):
    facts = load_mem().get(ctx["user_id"], {})
    words = query.lower().replace("_", " ").split()
    hits = {k: v for k, v in facts.items() if any(w in f"{k} {v}".lower().replace("_", " ") for w in words)}
    return {"status": "ok", "facts": hits or facts}

def book_trip(ctx, destination, amount_usd, cost_center):
    if not ctx["session"].get("approved"):                      # hard guard, not a prompt suggestion
        return {"status": "refused", "error": "No human approval on record for this session."}
    return {"status": "booked", "confirmation": "CONF-77821", "destination": destination,
            "amount_usd": amount_usd, "cost_center": cost_center}

TOOLS = {"save_memory": save_memory, "recall_memory": recall_memory, "book_trip": book_trip}
PAUSING_TOOLS = {"request_approval"}       # handled by the loop itself, not executed

def fn_schema(name, desc, props, required):
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": required}}}

TOOL_SCHEMAS = [
    fn_schema("save_memory", "Save a durable fact about the user (e.g. cost_center, diet).",
              {"key": {"type": "string"}, "value": {"type": "string"}}, ["key", "value"]),
    fn_schema("recall_memory", "Look up saved facts about the user from past conversations.",
              {"query": {"type": "string"}}, ["query"]),
    fn_schema("request_approval", "Ask a human manager to approve a spend. REQUIRED for any amount over 1000 USD.",
              {"purpose": {"type": "string"}, "amount_usd": {"type": "number"}}, ["purpose", "amount_usd"]),
    fn_schema("book_trip", "Book the trip. Only after approval status is approved.",
              {"destination": {"type": "string"}, "amount_usd": {"type": "number"}, "cost_center": {"type": "string"}},
              ["destination", "amount_usd", "cost_center"]),
]

SYSTEM = """You are a travel booking assistant.
- When the user states their cost center or dietary preference, call save_memory (keys: cost_center, diet).
- If the user refers to past conversations, call recall_memory first and answer from the result.
- Trips over 1000 USD: call request_approval BEFORE book_trip. Only book after approval. If rejected, do not book.
- Use the cost center the user gave you. Never invent one."""

## 2. Agent loop with pause, resume, and a scripted mock LLM

In [ ]:
def assistant_msg(content, calls):
    return {"role": "assistant", "content": content,
            "tool_calls": [{"id": c["id"], "type": "function",
                            "function": {"name": c["name"], "arguments": json.dumps(c["args"])}} for c in calls]}

def tool_msg(call_id, result):
    return {"role": "tool", "tool_call_id": call_id, "content": json.dumps(result)}

def history(messages):
    """[(tool_name, result_dict), ...] in order. Used by the mock LLM and by the tests."""
    id2name, out = {}, []
    for m in messages:
        if m["role"] == "assistant":
            for c in m.get("tool_calls") or []:
                id2name[c["id"]] = c["function"]["name"]
        elif m["role"] == "tool":
            out.append((id2name[m["tool_call_id"]], json.loads(m["content"])))
    return out

def ollama_llm(messages, tools):
    r = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)
    m = r.choices[0].message
    calls = [{"id": c.id, "name": c.function.name, "args": json.loads(c.function.arguments or "{}")}
             for c in (m.tool_calls or [])]
    return m.content, calls


def mock_llm(messages, tools):
    h = history(messages); names = [n for n, _ in h]; res = {n: r for n, r in h}
    first_user = messages[1]["content"]
    def call(name, args): return None, [{"id": f"mock-{len(h)}", "name": name, "args": args}]
    if "Berlin" in first_user:                                   # scenario A: new booking
        n_saved = names.count("save_memory")
        if n_saved == 0: return call("save_memory", {"key": "cost_center", "value": "CC-4471"})
        if n_saved == 1: return call("save_memory", {"key": "diet", "value": "vegetarian"})
        if "request_approval" not in names:
            return call("request_approval", {"purpose": "Berlin conference trip", "amount_usd": 1800})
        if res["request_approval"].get("status") == "rejected":
            return "Not booked: the approval was rejected.", []
        if "book_trip" not in names:
            return call("book_trip", {"destination": "Berlin", "amount_usd": 1800, "cost_center": "CC-4471"})
        return f"Booked Berlin, confirmation {res['book_trip']['confirmation']}, charged to CC-4471.", []
    if "recall_memory" not in names:                             # scenario B: returning user
        return call("recall_memory", {"query": "cost center diet"})
    f = res["recall_memory"]["facts"]
    return f"Your cost center is {f.get('cost_center')} and you are {f.get('diet')}.", []

def get_llm():
    return {"mock": mock_llm, "ollama": ollama_llm}[BACKEND]

def new_session(sid, user_id, user_msg):
    s = {"id": sid, "user_id": user_id, "status": "running", "pending": None, "approved": False,
         "final": None, "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]}
    save_session(s)
    return step(s)

def step(s, max_steps=12):
    """Run until the model gives a final answer OR asks for approval (then persist and return)."""
    for _ in range(max_steps):
        content, calls = get_llm()(s["messages"], TOOL_SCHEMAS)
        if not calls:
            s["messages"].append({"role": "assistant", "content": content})
            s.update(status="done", final=content); save_session(s); return s
        s["messages"].append(assistant_msg(content, calls))
        ctx = {"user_id": s["user_id"], "session": s}
        approval = None
        for c in calls:
            if c["name"] in PAUSING_TOOLS:
                approval = c; continue                            # answer these last, after the human decides
            try:
                result = TOOLS[c["name"]](ctx, **c["args"])
            except Exception as e:
                result = {"status": "error", "error": f"{type(e).__name__}: {e}"}
            print(f"  {c['name']}({c['args']}) -> {result}")
            s["messages"].append(tool_msg(c["id"], result))
        if approval:
            s.update(status="paused", pending=approval); save_session(s)
            print(f"  PAUSED on {approval['name']}({approval['args']})")
            return s
    raise RuntimeError("max_steps reached")

def resume(sid, decision):
    """Reload the session from disk (works even after a restart), inject the human decision, continue."""
    s = load_session(sid)
    assert s["status"] == "paused", f"session is {s['status']}, not paused"
    s["approved"] = decision.get("status") == "approved"
    s["messages"].append(tool_msg(s["pending"]["id"], decision))
    s.update(status="running", pending=None)
    print(f"  RESUMED with decision: {decision}")
    return step(s)

def booked(s):
    return any(n == "book_trip" and r.get("status") == "booked" for n, r in history(s["messages"]))

## 3. Tests (mock model, deterministic)

In [ ]:
BACKEND = "mock"
USER = "user-42"
reset_storage()

print("--- Session A: request that needs approval ---")
s = new_session("trip-001", USER,
    "Hi, I'm Sam. My cost center is CC-4471 and I'm vegetarian. Please book my Berlin conference trip, 1800 USD.")

# TEST 1: the run paused instead of booking
assert s["status"] == "paused" and s["pending"]["name"] == "request_approval"
assert not booked(s), "booked before approval!"
assert os.path.exists(f"{SESS_DIR}/trip-001.json")
print("PASS  run paused at the approval gate and was saved to disk")

# TEST 2: the guard holds even if something calls book_trip directly without approval
r = book_trip({"user_id": USER, "session": load_session("trip-001")}, "Berlin", 1800, "CC-4471")
assert r["status"] == "refused"
print("PASS  book_trip refuses without approval")

# TEST 3: 'restart' the process, then resume from disk only
del s
print("\n--- Human approves; resume from disk ---")
s = resume("trip-001", {"status": "approved", "approver": "manager@example.com", "ticket_id": "TCK-1042"})
assert s["status"] == "done" and booked(s)
assert "CC-4471" in s["final"]
print("PASS  resumed run completed the booking after approval\nFINAL:", s["final"])

In [ ]:
print("--- Session B: returning user, brand-new session ---")
b = new_session("trip-002", USER, "Hi again! What cost center did I give you last time, and what are my food preferences?")

# TEST 4: recalled from memory.json, not from session A's conversation
assert b["status"] == "done"
assert "CC-4471" in b["final"] and "vegetarian" in b["final"].lower()
assert "trip-001" not in json.dumps(b["messages"])
print("PASS  returning user recalled from memory in a new session\nFINAL:", b["final"])

# TEST 5: a different user must NOT see Sam's memory
other = new_session("trip-003", "user-99", "Hi again! What cost center did I give you last time, and what are my food preferences?")
assert "CC-4471" not in other["final"]
print("PASS  memory is isolated per user")

# TEST 6: rejection path
print("\n--- Rejection path ---")
new_session("trip-004", USER, "Please book my Berlin conference trip, 1800 USD. Cost center CC-4471.")
r = resume("trip-004", {"status": "rejected", "reason": "over budget"})
assert r["status"] == "done" and not booked(r)
print("PASS  rejected approval means no booking\nFINAL:", r["final"])

print("\nOutcome met: paused-and-resumed run cleared an approval step and recalled a returning user from memory.")

## 4. Same flow with a real open-source model (Ollama)

In [ ]:
# Requires Ollama running + `ollama pull qwen2.5:7b`. Small models can be flaky; re-run or try llama3.1:8b.
BACKEND = "ollama"
reset_storage()
try:
    print("--- Session A ---")
    s = new_session("real-001", "sam",
        "Hi, I'm Sam. My cost center is CC-4471 and I'm vegetarian. Please book my Berlin conference trip, 1800 USD.")
    print("status after turn 1:", s["status"], "| booked:", booked(s))
    if s["status"] == "paused":
        s = resume("real-001", {"status": "approved", "approver": "manager@example.com"})
        print("status after resume:", s["status"], "| booked:", booked(s), "\nFINAL:", s["final"])

    print("\n--- Session B (new session) ---")
    b = new_session("real-002", "sam", "Hi again! What cost center and food preferences do you have for me?")
    print("FINAL:", b["final"])
    print("\nPASS" if b["final"] and "CC-4471" in b["final"] else "\nFAIL", "recall from memory")
    print("Memory file:", json.dumps(load_mem(), indent=1))
except Exception as e:
    print("Ollama run failed:", type(e).__name__, e)

## What to screenshot for submission
The `PAUSED`, `RESUMED` and `PASS` lines from the mock run, plus the Session B answer and `memory.json`.

## How this maps to production
- `sessions/<id>.json` is what a durable session store (database, Vertex AI Agent Engine Sessions) does for you.
- `memory.json` is what a managed memory service (Vertex AI Memory Bank) does, minus the LLM-based fact extraction.
- The pause/resume contract is the same as ADK's long-running tools: return a pending marker, persist, and resume later with the result tied to the original call id.